In [5]:
import os
import pandas as pd
import numpy as np
import kagglehub
from google.colab import files

print("Downloading F1 World Championship dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")

# Load required dataset sheets
results = pd.read_csv(os.path.join(dataset_path, "results.csv"))
constructors = pd.read_csv(os.path.join(dataset_path, "constructors.csv"))
races = pd.read_csv(os.path.join(dataset_path, "races.csv"))
status = pd.read_csv(os.path.join(dataset_path, "status.csv"))
pit_stops = pd.read_csv(os.path.join(dataset_path, "pit_stops.csv"))

print("Data tables loaded. Beginning advanced feature aggregations...\n")


# 1. Total Driver Wins (total_matches_win)
driver_wins = results[results['positionOrder'] == 1].groupby('driverId').size().reset_index(name='total_matches_win')

# 2. Total Team Wins (team_no_of_wins)
team_wins = results[results['positionOrder'] == 1].groupby('constructorId').size().reset_index(name='team_no_of_wins')

# 3. Maximum Laps in a Race (highest_laps)
highest_laps_race = results.groupby('raceId')['laps'].max().reset_index(name='highest_laps')

# 4. Total Historical Laps Completed by a Team (total_laps)
total_team_laps = results.groupby('constructorId')['laps'].sum().reset_index(name='total_laps')

# 5. Unique Drivers per Team per Year (no_of_drivers)
results_with_year = pd.merge(results, races[['raceId', 'year']], on='raceId', how='left')
drivers_per_team_year = results_with_year.groupby(['constructorId', 'year'])['driverId'].nunique().reset_index(name='no_of_drivers')

# 6. Total Pitstops Per Driver Per Race (pitstop)
pit_counts = pit_stops.groupby(['raceId', 'driverId'])['stop'].max().reset_index(name='pitstop')


print("Merging data layers into single custom matrix...")

# Start with core results data
df_custom = results.copy()

# Join Constructor names
df_custom = pd.merge(df_custom, constructors[['constructorId', 'name']], on='constructorId', how='left')
df_custom = df_custom.rename(columns={'name': 'team_name'})

# Deriving the Car Engine feature from historical constructor entries
df_custom['car_engine'] = df_custom['team_name'].apply(lambda x: x.split('-')[1] if '-' in str(x) else x)

# Join Race metadata
df_custom = pd.merge(df_custom, races[['raceId', 'year', 'round', 'name', 'date']], on='raceId', how='left')

# Join Driver status descriptions
df_custom = pd.merge(df_custom, status[['statusId', 'status']], on='statusId', how='left')

# Join pre-computed statistical tables
df_custom = pd.merge(df_custom, driver_wins, on='driverId', how='left').fillna({'total_matches_win': 0})
df_custom = pd.merge(df_custom, team_wins, on='constructorId', how='left').fillna({'team_no_of_wins': 0})
df_custom = pd.merge(df_custom, highest_laps_race, on='raceId', how='left')
df_custom = pd.merge(df_custom, total_team_laps, on='constructorId', how='left')
df_custom = pd.merge(df_custom, drivers_per_team_year, on=['constructorId', 'year'], how='left')
df_custom = pd.merge(df_custom, pit_counts, on=['raceId', 'driverId'], how='left').fillna({'pitstop': 0})


requested_columns = [
    'team_name', 'car_engine', 'no_of_drivers', 'highest_laps', 'total_laps',
    'points', 'raceId', 'year', 'round', 'name', 'date', 'total_matches_win',
    'grid', 'positionOrder', 'laps', 'milliseconds', 'pitstop', 'status', 'team_no_of_wins'
]
df_custom = df_custom[requested_columns]

# Drop exact duplicate records
df_custom = df_custom.drop_duplicates()

# Coerce non-numeric string placeholders (like '\N' text values) to true NaN elements
df_custom['milliseconds'] = pd.to_numeric(df_custom['milliseconds'], errors='coerce')
df_custom['date'] = pd.to_datetime(df_custom['date'], errors='coerce')

# Standardize headers (strip whitespace, lowercase letters, replace internal spaces with underscores)
df_custom.columns = df_custom.columns.str.strip().str.lower().str.replace(' ', '_')

print("\n=== CLEANED CUSTOM DATA MATRIX PROFILE ===")
print("Shape of compiled data:", df_custom.shape)
print("\nMissing Values Sorted (Sourced from Missing Data Traps):")
print(df_custom.isnull().sum().sort_values(ascending=False).head(5))

print("\nFirst 5 Sample Records:")
display(df_custom.head())

# Save dataset to file system and dispatch down to user's computer
output_filename = "custom_f1_master_dataset.csv"
df_custom.to_csv(output_filename, index=False)
print(f"\n[SUCCESS] Compiled table saved. Downloading '{output_filename}' now...")
files.download(output_filename)

Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.
Data tables loaded. Beginning advanced feature aggregations...

Merging data layers into single custom matrix...

=== CLEANED CUSTOM DATA MATRIX PROFILE ===
Shape of compiled data: (26693, 19)

Missing Values Sorted (Sourced from Missing Data Traps):
milliseconds     19022
car_engine           0
team_name            0
no_of_drivers        0
highest_laps         0
dtype: int64

First 5 Sample Records:


,team_name,car_engine,no_of_drivers,highest_laps,total_laps,points,raceid,year,round,name,date,total_matches_win,grid,positionorder,laps,milliseconds,pitstop,status,team_no_of_wins
0,McLaren,McLaren,2,58,97845,10.0,18,2008,1,Australian Grand Prix,2008-03-16,105.0,1,1,58,5690616.0,0.0,Finished,185.0
1,BMW Sauber,BMW Sauber,2,58,7938,8.0,18,2008,1,Australian Grand Prix,2008-03-16,0.0,5,2,58,5696094.0,0.0,Finished,1.0
2,Williams,Williams,2,58,85881,6.0,18,2008,1,Australian Grand Prix,2008-03-16,23.0,7,3,58,5698779.0,0.0,Finished,114.0
3,Renault,Renault,2,58,38339,5.0,18,2008,1,Australian Grand Prix,2008-03-16,32.0,11,4,58,5707797.0,0.0,Finished,35.0
4,McLaren,McLaren,2,58,97845,4.0,18,2008,1,Australian Grand Prix,2008-03-16,1.0,3,5,58,5708630.0,0.0,Finished,185.0



[SUCCESS] Compiled table saved. Downloading 'custom_f1_master_dataset.csv' now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>